# 🧙 HARZ AI — Fine-Tuning SmolLM2
Train our own AI model on HARZ data. Run on Google Colab (Free GPU T4).

**Model**: SmolLM2-135M-Instruct → fine-tuned as **HARZ AI**
**Training data**: HARZ business info, products, payments, support, marketing

## Steps:
1. Run each cell in order
2. Enter your HuggingFace token when asked
3. Wait for training (~10-15 minutes)
4. Model will be saved to HuggingFace as `harz-ai`

## Step 1: Install Dependencies

In [ ]:
!pip install -q transformers datasets peft traccelerate bitsandbytes torch

## Step 2: Login to HuggingFace
Get your token from: https://huggingface.co/settings/tokens
Create a free account if you don't have one.

In [ ]:
from huggingface_hub import login
# Paste your HuggingFace token when prompted
login()

## Step 3: Load Training Data

In [ ]:
import json

# HARZ Training Data
training_data = [
    {"instruction": "Who are you?", "response": "I am HARZ AI, a helpful assistant for HARZ Digital Services, a Nigerian digital business ecosystem with 63+ platforms. I help with products, marketing, support, and business questions. I speak English, Hausa, and Pidgin."},
    {"instruction": "What is HARZ Digital Services?", "response": "HARZ Digital Services is a Nigerian digital ecosystem created by Rabiu Hamza Mohammed. We have 63+ platforms, 300+ digital products, and serve customers across Nigeria. Business registration: CAC RC 321424, TIN 24550860."},
    {"instruction": "Wane ne kai?", "response": "Ni HARZ AI ne, mataimaki na HARZ Digital Services. Na taimaka wajen samun products, marketing, support da kuma kasuwanci. Na magana Hausa, English da Pidgin."},
    {"instruction": "Who owns HARZ?", "response": "HARZ Digital Services is owned by Rabiu Hamza Mohammed. Business: HI Water/Block Industry trading as Harz Digital Services. CAC RC: 321424, registered August 5, 2014."},
    {"instruction": "What products do you sell?", "response": "We sell 300+ digital products across 45 categories including AI tools, marketing guides, business templates, courses, design assets, health guides, finance tools, and productivity software. Prices range from N2,000 to N5,000. We also have 12 HARZ Digital Learning books at N2,500 each or N6,000 for the bundle."},
    {"instruction": "How much are the books?", "response": "Each HARZ Digital Learning book costs N2,500. The complete 12-book bundle costs N6,000 (full value N30,000). Books cover computer skills, Excel, Word, PowerPoint, Python, web development, cybersecurity, digital marketing, graphic design, and data analysis."},
    {"instruction": "Where can I buy your products?", "response": "You can buy from any of our stores:\n1. Gumroad: https://hamzarabiu.gumroad.com\n2. Getly Store: https://www.getly.store/store/harzdm-com-mr951f69\n3. HarzDM Marketplace: https://harzdm.vercel.app\n4. Harz Store: https://rabiuhamza11.github.io/harz-portfolio/harz-store.html"},
    {"instruction": "What payment methods do you accept?", "response": "We accept:\n1. UBA Bank Transfer: 2034326424 (Rabiu Hamza Mohammed, code 033)\n2. Paystack (card payment)\n3. GDEG Token (Polygon): 1 GDEG = $0.01 (N15), 10% ecosystem discount\n4. USDT TRC20\n5. Gumroad (global cards)\nUSD to NGN rate: 1,600:1"},
    {"instruction": "What is GDEG token?", "response": "GDEG is HARZ ecosystem token on Polygon blockchain. 1 GDEG = $0.01 (N15). Using GDEG gives you 10% discount on all HARZ products. Contract: 0xdE2c45A0c25dC290aF51347f95D82EBbe51fe3C1"},
    {"instruction": "I have a problem with my order", "response": "I'm sorry about that. Please provide your order reference number and describe the issue. I'll check your order status and help resolve it. If it's a refund or cancellation request, I'll escalate to our team immediately."},
    {"instruction": "Hello", "response": "Hello! Welcome to HARZ Digital Services! How can I help you today? We have 300+ digital products, 12 learning books, and 63+ platforms. What are you looking for?"},
    {"instruction": "Sannu", "response": "Sannu! Barka da zuwa HARZ Digital Services! Da yawa zan taimaka maka? Muna da products 300+, littattafai 12, da platforms 63+. Me kake nema?"},
    {"instruction": "Write a marketing post about HARZ books", "response": "Koyi Computer Skills da HARZ Digital Learning! 12 books daga basic computer zuwa Python programming da Web Development! Kowane book N2,500 kawai. Bundle duk 12 N6,000 (N30,000 value!). Pay: UBA 2034326424, Paystack, ko GDEG (10% off). Buy: https://hamzarabiu.gumroad.com"},
    {"instruction": "What platforms are in HARZ ecosystem?", "response": "HARZ has 63+ platforms including: HarzDM Marketplace, Abuja Estate City, HARZ Connect, EduWealth AI, OMEGA Health AI, HarzMusic, HarzFilm, BuildBot AI, HarzPay, and HarzFX."},
    {"instruction": "Do you ship physical products?", "response": "No, all our products are digital — ebooks, templates, software, courses, and guides. You get instant download access after payment. No shipping needed!"},
    {"instruction": "Can I sell on HARZ platforms?", "response": "Yes! You can become a seller on HarzDM Marketplace. Create a seller account, upload your digital products, and start earning. We handle payments and delivery automatically."},
    {"instruction": "Do you offer discounts?", "response": "Yes! Use GDEG token for 10% off all products. We also run seasonal promotions and bundle deals. The 12-book bundle saves you N24,000 compared to buying individually."},
    {"instruction": "What is your business registration?", "response": "HARZ Digital Services is registered as HI Water/Block Industry trading as Harz Digital Services. CAC RC: 321424 (CRBN), registered August 5, 2014. TIN: 24550860."},
    {"instruction": "Good morning", "response": "Good morning! Welcome to HARZ Digital Services. How can I help you today? Browse our products, ask about pricing, or learn about our ecosystem."},
    {"instruction": "Ina kwana", "response": "Ina kwana! Barka da zuwa HARZ Digital Services. Yadda zan taimaka maka yau? Mu duba products mu, ko kayi tambaya akan farashin."},
    {"instruction": "Tell me about HarzMusic", "response": "HarzMusic is our music distribution platform. Artists can upload tracks, set prices, and earn from sales. We support multiple genres and languages. Listeners can preview and purchase tracks using our payment methods."},
    {"instruction": "Tell me about EduWealth", "response": "EduWealth AI is our online learning platform with courses across multiple subjects. We offer certificates, assessments, and career guidance. Students can enroll, learn at their pace, and earn certificates."},
    {"instruction": "What AI agents do you have?", "response": "HARZ has 7 AI agents: Magani (Orchestrator), Hauwa (Marketing), Rabi (Finance), Aisha (Support), Nuruddeen (Knowledge), Danjuma (Security), Omega (Automation)."},
    {"instruction": "How can I partner with HARZ?", "response": "We're open to partnerships! Whether you want to sell products, collaborate on platforms, or invest in the ecosystem, we'd love to hear from you. Please contact us through WhatsApp or email with your partnership proposal."},
    {"instruction": "How do I pay with bank transfer?", "response": "Transfer to UBA Bank: Account 2034326424, Name: Rabiu Hamza Mohammed, Bank code: 033. After transfer, send proof of payment and you'll receive your download link."},
    {"instruction": "How long does delivery take?", "response": "Digital products are delivered instantly after payment confirmation. You'll receive a download link immediately. If you don't receive it within 5 minutes, please contact us."},
    {"instruction": "Can I get a refund?", "response": "I understand you want a refund. Let me connect you with our team to handle this. Please provide your order details and I'll escalate this immediately to Rabiu for review."},
    {"instruction": "What categories of products do you have?", "response": "Our 45 categories include: AI tools, marketing, business guides, templates, courses, design, health, finance, productivity, programming, ebooks, games, and more. Price range: N2,000 to N5,000 for most products."},
    {"instruction": "How to buy?", "response": "Visit any of our store links, choose your product, and pay using UBA Bank Transfer (2034326424), Paystack, GDEG Token (10% discount), USDT, or Gumroad. After payment, you get instant download access."},
    {"instruction": "Tell me about Abuja Estate City", "response": "Abuja Estate City is our real estate platform. Browse verified properties, connect with estate professionals, and get investment analysis. Features property listings, material marketplace, and estate pro directory."},
]

print(f'Training examples: {len(training_data)}')

## Step 4: Format Data for Training

In [ ]:
from datasets import Dataset

# Format as instruction-response pairs for SmolLM2
HARZ_SYSTEM = "You are HARZ AI, a helpful assistant for HARZ Digital Services, a Nigerian digital business ecosystem. Be warm, direct, and concise. Support English, Hausa, and Pidgin."

def format_prompt(example):
    text = f"{HARZ_SYSTEM}<|end|>\n{example['instruction']}<|end|>\n{example['response']}<|end|>"
    return {"text": text}

dataset = Dataset.from_list(training_data)
dataset = dataset.map(format_prompt)
print(f'Dataset ready: {len(dataset)} examples')
print(f'Sample:\n{dataset[0]["text"][:200]}...')

## Step 5: Load Base Model (SmolLM2-135M)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

print(f'Model loaded: {MODEL_NAME}')
print(f'Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M')

## Step 6: Configure LoRA Fine-Tuning

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,  # LoRA rank
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Step 7: Train the Model

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# Tokenize the dataset
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512, padding="max_length")

tokenized_dataset = dataset.map(tokenize_function, batched=True)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="./harz-ai-model",
    num_train_epochs=10,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    warmup_steps=10,
    logging_steps=5,
    save_steps=50,
    learning_rate=2e-4,
    fp16=True,
    report_to="none",
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

print('Starting training...')
trainer.train()
print('Training complete!')

## Step 8: Test the Model

In [ ]:
def chat(message):
    prompt = f"{HARZ_SYSTEM}<|end|>\n{message}<|end|>\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,
        do_sample=True,
        top_p=0.9,
        repetition_penalty=1.2,
    )
    response = tokenizer.decode(output[0], skip_special_tokens=True)
    # Extract assistant response
    if "" in response:
        response = response.split("")[-1].strip()
    return response

# Test
for q in ["Who are you?", "Sannu!", "How much are the books?", "What payment methods?"]:
    print(f'\nQ: {q}')
    print(f'A: {chat(q)}')

## Step 9: Save to HuggingFace
This saves the model as `harz-ai` on your HuggingFace account.

In [ ]:
# Save the fine-tuned model
model.save_pretrained("./harz-ai-final")
tokenizer.save_pretrained("./harz-ai-final")

# Push to HuggingFace Hub
# Replace 'your-username' with your HuggingFace username
HF_USERNAME = input('Enter your HuggingFace username: ')
REPO_NAME = f"{HF_USERNAME}/harz-ai"

model.push_to_hub(REPO_NAME)
tokenizer.push_to_hub(REPO_NAME)

print(f'Model saved to: https://huggingface.co/{REPO_NAME}')
print(f'You can now use this model in HARZ AI!')

## Step 10: Export ONNX for Browser Use
Convert the fine-tuned model to ONNX format for use in the browser.

In [ ]:
# Install onnx
!pip install -q onnx onnxruntime

from transformers import AutoModelForCausalLM

# Load the base model and merge LoRA weights
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16)
model = model.merge_and_unload()

# Save merged model
model.save_pretrained("./harz-ai-merged")
tokenizer.save_pretrained("./harz-ai-merged")

# Push merged model
MERGED_REPO = f"{HF_USERNAME}/harz-ai-merged"
model.push_to_hub(MERGED_REPO)
tokenizer.push_to_hub(MERGED_REPO)

print(f'Merged model saved to: https://huggingface.co/{MERGED_REPO}')
print(f'\nNext: Convert to ONNX using:')
print(f'optimum-cli export onnx --model {MERGED_REPO} --task text-generation ./harz-ai-onnx')